In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import ChatMessage, ChatMessageRole

w = WorkspaceClient()

def ask_reviews(question, k=5):
    # 1️⃣ RETRIEVE — semantic search for relevant reviews
    res = w.vector_search_indexes.query_index(
        index_name="northstar.gold.reviews_index",
        columns=["review_comment_message"],
        query_text=question,
        num_results=k)
    reviews = "\n".join(f"- {row[0]}" for row in res.result.data_array)

    # 2️⃣ AUGMENT — build a grounded prompt
    prompt = f"""You are a customer insights analyst.
Based ONLY on these customer reviews (they are in Portuguese), answer the question in English.

Reviews:
{reviews}

Question: {question}"""

    # 3️⃣ GENERATE — call the LLM
    resp = w.serving_endpoints.query(
        name="databricks-meta-llama-3-1-8b-instruct",          # any chat model from your system.ai list
        messages=[ChatMessage(role=ChatMessageRole.USER, content=prompt)])
    return resp.choices[0].message.content

# Ask it a question!
print(ask_reviews("What are the main complaints about damaged products?"))

In [0]:
print(ask_reviews("Are customers happy with product quality? Give examples."))

In [0]:
print(ask_reviews("What should we fix first to improve satisfaction?"))